# STIR-Net V1 — 16 split-identity localization

Notebook 15 established that the **full five-frame temporal information is present and connected**, but source-9 split hypotheses still use almost identical temporal attention.

This notebook does **no training**. It loads the final Notebook-15 joint checkpoint and localizes *where* split identity collapses.

## Questions

For source 9, this notebook traces:

```text
current-component embedding
        ↓
component temporal fusion
        ↓
split-slot embedding
        ↓
QueryBuilder output
        ↓
decoder self-attention
        ↓
temporal Q projection
        ↓
content Q·K logits
        +
physical/temporal relation bias
        ↓
temporal softmax
        ↓
node / tracklet messages and gates
        ↓
spatial cross-attention
        ↓
FFN
        ↓
center update
```

It also audits the **detection GNN candidate-edge softmax** to determine whether the all-pairs GNN itself is nearly uniform.

## Main diagnostics

1. Magnitude of the learned split-slot identity relative to the shared component token.
2. Pairwise sibling similarity after every decoder sub-operation.
3. Temporal attention decomposed into:
   - content-only `Q·K`,
   - relation-bias-only,
   - combined production attention.
4. Node-message and residual-gate magnitudes.
5. Counterfactuals:
   - skip decoder self-attention before temporal reading;
   - scale the learned split-slot offsets without retraining.
6. Step-50 → step-85 parameter movement for split-slot and temporal-reader parameters.
7. Detection-GNN edge-attention entropy, accepted-edge mass, and source-9-related node diagnostics.

The notebook is diagnostic only. It does **not** edit the model or overwrite checkpoints.

In [ ]:
from pathlib import Path
from dataclasses import replace
import gc
import json
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from learned.stirnet import StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.model.coordinates import feature_grid_coordinates_um
from learned.stirnet.model.graph_encoder import segment_softmax
from learned.stirnet.model.query_builder import (
    QUERY_PRIMARY,
    QUERY_SPLIT,
)
from learned.stirnet.model.query_decoder import _cap_feature_tokens
from learned.stirnet.training.checkpoint import load_checkpoint
from learned.stirnet.training.trainer import move_to_device

SEED = 40266
SOURCE_ID = 9
AMP_DTYPE = torch.float16

REPO_ROOT = _repo_root(Path.cwd())
DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)
NB15_RUN = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "15_hierarchical_temporal_memory"
)
CHECKPOINT_STEP50 = NB15_RUN / "checkpoint_temporal_dense.pt"
CHECKPOINT_STEP85 = NB15_RUN / "checkpoint_joint.pt"

RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "16_split_identity_localization"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 16 requires CUDA.")

device = torch.device("cuda")

for path in (CHECKPOINT_STEP50, CHECKPOINT_STEP85):
    if not path.exists():
        raise FileNotFoundError(
            f"Required Notebook-15 checkpoint not found:\n{path}\n"
            "Run Notebook 15 completely first."
        )

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("NB15 run   :", NB15_RUN)
print("Run dir    :", RUN_DIR)
print("GPU        :", torch.cuda.get_device_name(0))
print("Step 50    :", CHECKPOINT_STEP50)
print("Step 85    :", CHECKPOINT_STEP85)

## 1. Load the exact real scene and the final step-85 model

We use the same full all-cell scene. No cell patch is isolated.

In [ ]:
batch, sample = build_real_batch(DATA_DIR)
cfg = _reduced_config()

assert sample["current_count"] == 36
assert sample["target_count"] == 33
assert sample["temporal_tracklets"] == 52
assert sample["split_companions_by_source"].get(SOURCE_ID) == 8

def prepare_device_batch(cpu_batch):
    result = {}
    for key, value in cpu_batch.items():
        if key == "targets":
            result[key] = value
        elif key == "spatial_inputs":
            result[key] = value.to(
                device=device,
                dtype=AMP_DTYPE,
                non_blocking=True,
            )
        elif key == "instance_labels":
            result[key] = value.to(
                device=device,
                dtype=torch.int32,
                non_blocking=True,
            )
        else:
            result[key] = move_to_device(value, device)
    return result

b = prepare_device_batch(batch)

model = StirNet(cfg).to(device).eval()
ckpt85 = load_checkpoint(
    CHECKPOINT_STEP85,
    model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    map_location="cpu",
    strict=True,
    migrate_history=True,
)
model.eval()

N = int(b["graph_x"].shape[0])
E = int(b["graph_edge_index"].shape[1])

print("Scene:", sample)
print("Nodes:", N)
print("Candidate edges:", E)
print("Expected all-pairs:", N * (N - 1))
print("Tracklets:", len(b["temporal_ref_um"]))
print("History valid:", int(b["node_history_valid"].sum()), "/", N)
print("Loaded step:", ckpt85.get("step"))

assert E == N * (N - 1)

## 2. General representation diagnostics

The same summary functions will be used at every sub-operation.

For a group of sibling tokens:

- cosine near `1` means directions are almost identical;
- small pairwise L2 means their actual vectors are almost identical;
- small across-slot standard deviation means little identity information remains.

In [ ]:
def offdiag_values(matrix):
    matrix = torch.as_tensor(matrix)
    if matrix.shape[0] < 2:
        return matrix.new_zeros((0,))
    mask = ~torch.eye(
        matrix.shape[0],
        dtype=torch.bool,
        device=matrix.device,
    )
    return matrix[mask]

def tensor_group_summary(x):
    x = torch.as_tensor(x).float()
    if x.ndim != 2:
        x = x.flatten(1)
    if x.shape[0] < 2:
        return {
            "cos_mean": float("nan"),
            "cos_median": float("nan"),
            "pair_l2_mean": float("nan"),
            "pair_l2_median": float("nan"),
            "slot_std_rms": float("nan"),
            "token_rms": float(x.square().mean().sqrt()) if x.numel() else float("nan"),
            "slot_std_to_token_rms": float("nan"),
        }

    cosine = F.normalize(x, dim=-1, eps=1e-8) @ F.normalize(x, dim=-1, eps=1e-8).T
    cos_values = offdiag_values(cosine)
    pair_l2 = torch.pdist(x)
    slot_std_rms = x.std(dim=0, unbiased=False).square().mean().sqrt()
    token_rms = x.square().mean().sqrt()

    return {
        "cos_mean": float(cos_values.mean()),
        "cos_median": float(cos_values.median()),
        "pair_l2_mean": float(pair_l2.mean()),
        "pair_l2_median": float(pair_l2.median()),
        "slot_std_rms": float(slot_std_rms),
        "token_rms": float(token_rms),
        "slot_std_to_token_rms": float(slot_std_rms / token_rms.clamp_min(1e-12)),
    }

def distribution_group_summary(weights):
    p = torch.as_tensor(weights).float()
    if p.ndim == 3:
        p = p.mean(dim=0)
    p = p / p.sum(dim=-1, keepdim=True).clamp_min(1e-12)

    result = tensor_group_summary(p)
    entropy = -(p * p.clamp_min(1e-12).log()).sum(dim=-1)
    normalized_entropy = entropy / max(math.log(max(p.shape[-1], 2)), 1e-8)

    result.update({
        "entropy_mean": float(entropy.mean()),
        "normalized_entropy_mean": float(normalized_entropy.mean()),
        "max_weight_mean": float(p.max(dim=-1).values.mean()),
        "unique_top_keys": int(torch.unique(p.argmax(dim=-1)).numel()),
    })
    return result

def summarize_stage(name, tensor, query_indices, rows):
    selected = tensor[0, query_indices] if tensor.ndim == 3 else tensor[query_indices]
    summary = tensor_group_summary(selected.detach().float().cpu())
    rows.append({"stage": name, **summary})
    return summary

def source9_query_indices(qstate):
    types = qstate.query_types[0]
    sources = qstate.source_instance_ids[0]
    mask = (
        (sources == SOURCE_ID)
        & ((types == QUERY_PRIMARY) | (types == QUERY_SPLIT))
    )
    indices = torch.nonzero(mask, as_tuple=False).flatten()
    primary = indices[types[indices] == QUERY_PRIMARY]
    split = indices[types[indices] == QUERY_SPLIT]

    labels = {}
    for q in primary.tolist():
        labels[int(q)] = "primary"
    for slot, q in enumerate(split.tolist()):
        labels[int(q)] = f"split{slot}"

    return indices, primary, split, labels

def rowwise_l2(x):
    x = torch.as_tensor(x).float()
    return torch.linalg.vector_norm(x, dim=-1)

## 3. Reconstruct the model up to QueryBuilder

We explicitly retain:

- detection-GNN node memory;
- tracklet memory after CR1 / CR2;
- E3, E2 and D1 spatial features;
- the **raw current-component embedding before temporal fusion**;
- the fused component embedding;
- the actual QueryBuilder output.

This is the first place to ask whether split identity is already too small.

In [ ]:
@torch.no_grad()
def build_predecoder_state(model, b):
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
        acq = model.acquisition(
            b["spacing_um"],
            b["dref_um"],
        )
        pyramid = model.encoder(
            b["spatial_inputs"],
            b["spacing_um"],
            acq,
            b.get("spatial_padding_mask"),
        )

        temporal = model._build_temporal(
            b["graph_x"],
            b["graph_edge_index"],
            b["graph_edge_attr"],
            b["tracklet_id"],
            b["temporal_ref_um"],
            b["temporal_status"],
            b["hypothesis_edge_index"],
            b["hypothesis_edge_attr"],
            b["temporal_batch"],
            b["dref_um"],
            node_instance_grid=b.get("node_instance_grid"),
            node_history_valid=b.get("node_history_valid"),
            history_support=b.get("history_support"),
            history_support_valid=b.get("history_support_valid"),
            history_support_dt=b.get("history_support_dt"),
            history_support_center_um=b.get("history_support_center_um"),
            history_support_extent_um=b.get("history_support_extent_um"),
            best_current_component_id=b.get("best_current_component_id"),
            best_component_overlap=b.get("best_component_overlap"),
            second_best_component_overlap=b.get("second_best_component_overlap"),
            node_observed_ref_um=b.get("node_observed_ref_um"),
            node_time_offset=b.get("node_time_offset"),
            node_ids=b.get("node_ids"),
        )

        e3, temporal = model.cr1(
            pyramid.features[3],
            pyramid.spacings_um[3],
            temporal,
            b["dref_um"],
            acq,
            pyramid.padding_masks[3] if pyramid.padding_masks else None,
        )
        e2 = model.decoder.decode_to_e2(
            e3,
            pyramid,
            acq,
        )
        e2, temporal = model.cr2(
            e2,
            pyramid.spacings_um[2],
            temporal,
            b["dref_um"],
            acq,
            pyramid.padding_masks[2] if pyramid.padding_masks else None,
        )
        d1, d0, mask_features = model.decoder.decode_from_e2(
            e2,
            pyramid,
            acq,
        )

        pooled = model.query_builder._pool_instances(
            e2,
            pyramid.spacings_um[2],
            b["instance_labels"],
            b["instance_ids"],
            b["instance_batch"],
            b["instance_centroids_um"],
        )
        raw_component = (
            model.query_builder.feature_proj(pooled)
            + model.query_builder.geom_proj(b["instance_features"])
        )

        fused_component, component_debug = model.query_builder.temporal_fusion(
            raw_component,
            b["instance_centroids_um"],
            b["instance_batch"],
            temporal,
            b["dref_um"],
            memory_ablation="full",
            return_debug=True,
            full_attention=True,
        )

        qstate = model.query_builder(
            e2,
            pyramid.spacings_um[2],
            b["instance_labels"],
            b["instance_features"],
            b["instance_ids"],
            b["instance_batch"],
            b["instance_centroids_um"],
            b["dref_um"],
            temporal,
            memory_ablation="full",
            return_debug=True,
            full_attention=True,
        )

    return {
        "acq": acq,
        "pyramid": pyramid,
        "temporal": temporal,
        "e3": e3,
        "e2": e2,
        "d1": d1,
        "d0": d0,
        "mask_features": mask_features,
        "raw_component": raw_component,
        "fused_component": fused_component,
        "component_debug": component_debug,
        "qstate": qstate,
    }

gc.collect()
torch.cuda.empty_cache()
state = build_predecoder_state(model, b)

qstate = state["qstate"]
source_indices, source_primary, source_splits, source_labels = source9_query_indices(qstate)

print("Source-9 seeded query indices:", source_indices.tolist())
print("Labels:", source_labels)
print("Initial source-9 references (cell scale):")
display(pd.DataFrame(
    qstate.references_cellscale[0, source_indices].float().cpu().numpy(),
    index=[source_labels[int(q)] for q in source_indices.tolist()],
    columns=["z", "y", "x"],
))

## 4. How large is the split-slot identity signal?

All source-9 siblings share the same component representation and initial centroid.

Their only explicit sibling identity at QueryBuilder output is the learned split-slot embedding (plus primary-vs-split type for the primary query).

This cell compares those magnitudes directly.

In [ ]:
instance_ids = b["instance_ids"]
component_row = int(
    torch.nonzero(instance_ids == SOURCE_ID, as_tuple=False).flatten()[0]
)

raw_c = state["raw_component"][component_row].float()
fused_c = state["fused_component"][component_row].float()
component_delta = fused_c - raw_c

slot_weights = model.query_builder.split_slot_embedding.weight.detach().float()
split_type = model.query_builder.type_embedding.weight[QUERY_SPLIT].detach().float()
primary_type = model.query_builder.type_embedding.weight[QUERY_PRIMARY].detach().float()

slot_pair_l2 = torch.pdist(slot_weights)
slot_summary = tensor_group_summary(slot_weights.cpu())

magnitude_df = pd.DataFrame([
    {"quantity": "raw source-9 component", "l2_norm": float(raw_c.norm()), "rms": float(raw_c.square().mean().sqrt())},
    {"quantity": "component temporal-fusion delta", "l2_norm": float(component_delta.norm()), "rms": float(component_delta.square().mean().sqrt())},
    {"quantity": "fused source-9 component", "l2_norm": float(fused_c.norm()), "rms": float(fused_c.square().mean().sqrt())},
    {"quantity": "primary type embedding", "l2_norm": float(primary_type.norm()), "rms": float(primary_type.square().mean().sqrt())},
    {"quantity": "split type embedding", "l2_norm": float(split_type.norm()), "rms": float(split_type.square().mean().sqrt())},
    {"quantity": "split-slot mean norm", "l2_norm": float(slot_weights.norm(dim=-1).mean()), "rms": float(slot_weights.square().mean().sqrt())},
    {"quantity": "split-slot mean pair distance", "l2_norm": float(slot_pair_l2.mean()), "rms": float("nan")},
])

print("Split-slot pairwise / directional summary:")
print(slot_summary)
display(magnitude_df)

initial_split_tokens = qstate.embeddings[0, source_splits].float().cpu()
print("Source-9 split tokens at QueryBuilder output:")
display(pd.DataFrame([tensor_group_summary(initial_split_tokens)]))

## 5. Did the query-stage training actually change the slot identities?

The step-50 checkpoint is ideal as a baseline because the query group was frozen through `temporal_dense`.

We compare step 50 → step 85 without loading a second GPU model.

In [ ]:
raw50 = torch.load(
    CHECKPOINT_STEP50,
    map_location="cpu",
    weights_only=False,
)["model"]
raw85 = torch.load(
    CHECKPOINT_STEP85,
    map_location="cpu",
    weights_only=False,
)["model"]

PARAMETER_KEYS = [
    "query_builder.split_slot_embedding.weight",
    "query_builder.type_embedding.weight",
    "query_builder.temporal_fusion.node_attention.q.weight",
    "query_builder.temporal_fusion.node_attention.k.weight",
    "query_builder.temporal_fusion.node_gate.weight",
    "query_decoder.layers.0.self_attn.in_proj_weight",
    "query_decoder.layers.0.temporal_fusion.node_attention.q.weight",
    "query_decoder.layers.0.temporal_fusion.node_attention.k.weight",
    "query_decoder.layers.0.temporal_fusion.node_gate.weight",
    "query_decoder.layers.1.temporal_fusion.node_attention.q.weight",
    "query_decoder.layers.2.temporal_fusion.node_attention.q.weight",
]

parameter_rows = []
for key in PARAMETER_KEYS:
    if key not in raw50 or key not in raw85:
        parameter_rows.append({
            "parameter": key,
            "found": False,
        })
        continue

    a = raw50[key].float()
    c = raw85[key].float()
    delta = c - a
    parameter_rows.append({
        "parameter": key,
        "found": True,
        "step50_norm": float(a.norm()),
        "step85_norm": float(c.norm()),
        "delta_norm": float(delta.norm()),
        "relative_delta": float(delta.norm() / a.norm().clamp_min(1e-12)),
        "max_abs_delta": float(delta.abs().max()),
    })

parameter_df = pd.DataFrame(parameter_rows)
parameter_df.to_csv(
    RUN_DIR / "parameter_movement_step50_to_step85.csv",
    index=False,
)
display(parameter_df)

## 6. Exact temporal-attention decomposition helper

This reproduces the current `TemporalMemoryAttention` math in evaluation mode and exposes intermediate tensors:

```text
normalized query
    ↓
Q projection
K projection
    ↓
content logits Q·K / sqrt(d)
    +
10-D learned relation bias
    ↓
combined logits
    ↓
softmax
    ↓
value-weighted message
    ↓
output projection
```

We can therefore determine whether sibling differences disappear before softmax, or whether they exist in content logits but are overwhelmed by another term.

In [ ]:
@torch.no_grad()
def trace_memory_attention(
    attention,
    query_tokens,
    query_ref_um,
    memory_tokens,
    observed_ref_um,
    projected_ref_um,
    time_offset,
    history_valid,
    dref,
):
    # Notebook 16 is intentionally a single-sample real diagnostic.
    qh = attention._split(
        attention.q(query_tokens)
    ).permute(1, 0, 2)
    kh = attention._split(
        attention.k(memory_tokens)
    ).permute(1, 0, 2)
    vh = attention._split(
        attention.v(memory_tokens)
    ).permute(1, 0, 2)

    content_logits = torch.einsum(
        "hqd,hkd->hqk",
        qh,
        kh,
    ).float() / math.sqrt(attention.head_dim)

    relation_bias = attention.relation_bias(
        query_ref_um,
        observed_ref_um,
        projected_ref_um,
        time_offset,
        history_valid,
        dref,
    ).permute(2, 0, 1).float()

    combined_logits = content_logits + relation_bias

    content_weights = torch.softmax(content_logits, dim=-1)
    relation_weights = torch.softmax(relation_bias, dim=-1)
    combined_weights = torch.softmax(combined_logits, dim=-1)

    message_pre_out = torch.einsum(
        "hqk,hkd->hqd",
        combined_weights,
        vh.float(),
    ).permute(1, 0, 2).reshape(
        query_tokens.shape[0],
        attention.d_model,
    )
    message_out = attention.out(
        message_pre_out.to(query_tokens.dtype)
    )

    return {
        "q_projected": qh.permute(1, 0, 2).reshape(query_tokens.shape[0], -1),
        "k_projected": kh.permute(1, 0, 2).reshape(memory_tokens.shape[0], -1),
        "content_logits": content_logits,
        "relation_bias": relation_bias,
        "combined_logits": combined_logits,
        "content_weights": content_weights,
        "relation_weights": relation_weights,
        "combined_weights": combined_weights,
        "message_pre_out": message_pre_out,
        "message_out": message_out,
    }

@torch.no_grad()
def trace_hierarchical_fusion(
    fusion,
    query_tokens,
    query_ref_um,
    temporal,
    dref_um,
):
    normalized = fusion.norm(query_tokens)
    output = query_tokens

    node = trace_memory_attention(
        fusion.node_attention,
        normalized,
        query_ref_um,
        temporal.node_memory.tokens,
        temporal.node_memory.observed_ref_um,
        temporal.node_memory.projected_ref_um,
        temporal.node_memory.time_offset,
        temporal.node_memory.history_valid,
        dref_um[0],
    )
    node_gate = torch.sigmoid(
        fusion.node_gate(
            torch.cat(
                [normalized, node["message_out"]],
                dim=-1,
            )
        )
    )
    after_node = output + node_gate * node["message_out"]

    if (
        temporal.history_support_valid is not None
        and temporal.history_support_valid.shape[0] == temporal.tokens.shape[0]
    ):
        tracklet_valid = temporal.history_support_valid.any(dim=-1)
    else:
        tracklet_valid = torch.ones(
            temporal.tokens.shape[0],
            device=temporal.tokens.device,
            dtype=torch.bool,
        )

    tracklet = trace_memory_attention(
        fusion.tracklet_attention,
        normalized,
        query_ref_um,
        temporal.tokens,
        temporal.ref_um,
        temporal.ref_um,
        temporal.ref_um.new_zeros(
            (temporal.tokens.shape[0],)
        ),
        tracklet_valid,
        dref_um[0],
    )
    tracklet_gate = torch.sigmoid(
        fusion.tracklet_gate(
            torch.cat(
                [normalized, tracklet["message_out"]],
                dim=-1,
            )
        )
    )
    after_tracklet = (
        after_node
        + tracklet_gate * tracklet["message_out"]
    )

    ffn_input = fusion.ffn_norm(after_tracklet)
    ffn_raw = fusion.ffn(ffn_input)
    ffn_scale = torch.sigmoid(fusion.ffn_gate)
    ffn_contribution = ffn_scale * ffn_raw
    final = after_tracklet + ffn_contribution

    return {
        "input": query_tokens,
        "normalized": normalized,
        "node": node,
        "node_gate": node_gate,
        "node_contribution": node_gate * node["message_out"],
        "after_node": after_node,
        "tracklet": tracklet,
        "tracklet_gate": tracklet_gate,
        "tracklet_contribution": tracklet_gate * tracklet["message_out"],
        "after_tracklet": after_tracklet,
        "ffn_input": ffn_input,
        "ffn_raw": ffn_raw,
        "ffn_scale": ffn_scale,
        "ffn_contribution": ffn_contribution,
        "final": final,
    }

def logit_term_summary(name, tensor, source_rows):
    # tensor: [H,Q,K]
    selected = tensor[:, source_rows].detach().float().cpu()
    flattened = selected.permute(1, 0, 2).flatten(1)
    group = tensor_group_summary(flattened)
    return {
        "term": name,
        **group,
        "absolute_mean": float(selected.abs().mean()),
        "std_all": float(selected.std(unbiased=False)),
        "range_mean_per_query": float(
            (selected.amax(dim=(0, 2)) - selected.amin(dim=(0, 2))).mean()
        ),
    }

## 7. Trace every decoder sub-operation

The forward pass below is mathematically equivalent to the current decoder in `eval()` mode, but stores intermediate states.

For each decoder layer it records:

- layer input;
- self-normalized input;
- self-attention contribution;
- self-attention residual;
- temporal normalized query;
- temporal Q projection;
- node content logits;
- relation bias;
- production temporal softmax;
- node / tracklet residual messages and gates;
- temporal-fusion output;
- spatial cross-attention contribution;
- spatial residual;
- FFN contribution;
- final query embedding;
- center update.

This is the core localization experiment.

In [ ]:
@torch.no_grad()
def trace_decoder(model, state, b):
    q = state["qstate"]
    temporal = state["temporal"]
    spatial_features = [
        state["e3"],
        state["e2"],
        state["d1"],
    ]
    spatial_spacings = [
        state["pyramid"].spacings_um[3],
        state["pyramid"].spacings_um[2],
        state["pyramid"].spacings_um[1],
    ]

    traces = []
    previous_mask = None

    for layer_index, layer in enumerate(model.query_decoder.layers):
        feat, spacing = _cap_feature_tokens(
            spatial_features[layer_index],
            spatial_spacings[layer_index],
            model.query_decoder.cfg.max_spatial_tokens,
        )
        proj = model.query_decoder.feature_proj[layer_index](feat)
        mask_feat = model.query_decoder.mask_feature_proj[layer_index](feat)

        B, _, Z, Y, X = proj.shape
        spatial_tokens = proj.flatten(2).transpose(1, 2)
        pos_um = feature_grid_coordinates_um(
            (Z, Y, X),
            spacing,
            relative_to_center=True,
        )

        support = model.query_decoder._reference_support(
            q,
            pos_um,
            b["dref_um"],
            layer_index,
        ).reshape(B, q.embeddings.shape[1], Z, Y, X)

        source_support = model.query_decoder._source_instance_support(
            q,
            b["instance_labels"],
            (Z, Y, X),
        )
        seeded = (
            (q.query_types == QUERY_PRIMARY)
            | (q.query_types == QUERY_SPLIT)
        )
        support = support | (
            source_support
            & seeded[..., None, None, None]
        )

        if previous_mask is not None:
            prev = F.interpolate(
                previous_mask.sigmoid(),
                size=(Z, Y, X),
                mode="trilinear",
                align_corners=False,
            )
            prev_support = (
                prev
                > model.query_decoder.cfg.mask_attention_threshold
            )
            prev_support = model.query_decoder._dilate(
                prev_support,
                spacing,
                b["dref_um"],
            )
            support = support | prev_support

        support_flat = support.flatten(2)

        x0 = q.embeddings
        self_norm = layer.self_norm(x0)
        self_message, self_weights = layer.self_attn(
            self_norm,
            self_norm,
            self_norm,
            key_padding_mask=q.padding_mask,
            need_weights=True,
            average_attn_weights=False,
        )
        after_self = x0 + self_message

        valid = ~q.padding_mask
        valid_rows = torch.nonzero(
            valid[0],
            as_tuple=False,
        ).flatten()

        refs_um_valid = (
            q.references_cellscale
            * b["dref_um"][:, None, None]
        )[valid]

        temporal_trace = trace_hierarchical_fusion(
            layer.temporal_fusion,
            after_self[valid],
            refs_um_valid,
            temporal,
            b["dref_um"],
        )

        after_temporal = after_self.clone()
        after_temporal[valid] = temporal_trace["final"]

        cross_norm = layer.cross_norm(after_temporal)
        spatial_message = layer.cross_attn(
            cross_norm,
            spatial_tokens,
            pos_um,
            q.references_cellscale
            * b["dref_um"][:, None, None],
            support_flat,
            q.padding_mask,
            b["dref_um"],
        )
        after_spatial = after_temporal + spatial_message

        ffn_input = layer.ffn_norm(after_spatial)
        ffn_message = layer.ffn(ffn_input)
        final_x = after_spatial + ffn_message
        final_x = final_x.masked_fill(
            q.padding_mask[..., None],
            0,
        )

        reference_before = q.references_cellscale
        raw_delta = layer.center(final_x)
        center_delta = layer._bounded_center_delta(
            raw_delta,
            q,
        )
        refs = reference_before + center_delta

        mask_embedding = layer.mask_embed(final_x)
        masks = torch.einsum(
            "bqc,bczyx->bqzyx",
            mask_embedding,
            mask_feat,
        )
        masks = masks.masked_fill(
            q.padding_mask[..., None, None, None],
            -20.0,
        )

        q_next = replace(
            q,
            embeddings=final_x,
            references_cellscale=refs,
        )

        traces.append({
            "layer": layer_index,
            "q_before": q,
            "valid_rows": valid_rows,
            "x_input": x0,
            "self_norm": self_norm,
            "self_message": self_message,
            "self_weights": self_weights,
            "after_self": after_self,
            "temporal": temporal_trace,
            "after_temporal": after_temporal,
            "cross_norm": cross_norm,
            "spatial_message": spatial_message,
            "after_spatial": after_spatial,
            "ffn_input": ffn_input,
            "ffn_message": ffn_message,
            "final_x": final_x,
            "reference_before": reference_before,
            "center_delta": center_delta,
            "references_after": refs,
            "coarse_mask_logits": masks,
            "support_count": support_flat.sum(dim=-1),
        })

        q = q_next
        previous_mask = masks

    return q, traces

gc.collect()
torch.cuda.empty_cache()

with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
    final_q, traces = trace_decoder(
        model,
        state,
        b,
    )

print("Traced decoder layers:", len(traces))

## 8. Where do sibling tokens become similar?

This is the first decisive table.

If QueryBuilder output is already nearly identical, the identity signal is too small at construction.

If it is distinct initially but becomes much more similar after self-attention, self-attention is homogenizing the split slots.

If temporal fusion restores diversity and later spatial/FFN removes it, the failure is downstream.

In [ ]:
representation_rows = []

summarize_stage(
    "query_builder_output",
    qstate.embeddings,
    source_splits,
    representation_rows,
)

for trace in traces:
    layer = trace["layer"]

    for name, tensor in [
        (f"L{layer}_self_norm", trace["self_norm"]),
        (f"L{layer}_self_message", trace["self_message"]),
        (f"L{layer}_after_self", trace["after_self"]),
        (f"L{layer}_after_temporal", trace["after_temporal"]),
        (f"L{layer}_spatial_message", trace["spatial_message"]),
        (f"L{layer}_after_spatial", trace["after_spatial"]),
        (f"L{layer}_ffn_message", trace["ffn_message"]),
        (f"L{layer}_final", trace["final_x"]),
    ]:
        summarize_stage(
            name,
            tensor,
            source_splits,
            representation_rows,
        )

representation_df = pd.DataFrame(
    representation_rows
)
representation_df.to_csv(
    RUN_DIR / "source9_suboperation_representation.csv",
    index=False,
)

display(representation_df)

## 9. Self-attention: does it mix the siblings into the same representation?

We measure:

- source-9 split-to-split attention mass;
- source-9 split-to-primary mass;
- attention entropy;
- sibling similarity before and after the self-attention residual.

In [ ]:
self_attention_rows = []

for trace in traces:
    layer = trace["layer"]
    weights = (
        trace["self_weights"][0]
        .detach()
        .float()
        .cpu()
    )  # [heads, Q, Q]

    split_idx = source_splits.detach().cpu()
    primary_idx = source_primary.detach().cpu()

    split_rows = weights[:, split_idx, :]
    mean_weights = split_rows.mean(dim=0)

    split_mass = mean_weights[:, split_idx].sum(dim=-1)
    primary_mass = (
        mean_weights[:, primary_idx].sum(dim=-1)
        if len(primary_idx)
        else torch.zeros(len(split_idx))
    )
    entropy = -(
        mean_weights
        * mean_weights.clamp_min(1e-12).log()
    ).sum(dim=-1)

    self_attention_rows.append({
        "layer": layer,
        "mean_split_to_split_mass": float(split_mass.mean()),
        "mean_split_to_primary_mass": float(primary_mass.mean()),
        "mean_entropy": float(entropy.mean()),
        "normalized_entropy": float(
            entropy.mean()
            / math.log(mean_weights.shape[-1])
        ),
        "unique_top_queries": int(
            torch.unique(mean_weights.argmax(dim=-1)).numel()
        ),
    })

self_attention_df = pd.DataFrame(
    self_attention_rows
)
self_attention_df.to_csv(
    RUN_DIR / "source9_self_attention.csv",
    index=False,
)
display(self_attention_df)

## 10. Temporal query projection and logits: content vs relation bias

This is the second decisive table.

For each decoder layer we compare sibling diversity in:

- the normalized temporal-reader input;
- Q projections;
- content logits;
- relation bias;
- combined logits;
- content-only attention;
- relation-only attention;
- production combined attention.

The physical relation bias should be nearly identical for siblings while their references are co-located. The question is whether the learned content term has enough identity variation to overcome that shared geometry.

In [ ]:
temporal_rows = []

for trace in traces:
    layer = trace["layer"]
    valid_rows = trace["valid_rows"].detach().cpu()
    global_to_valid = {
        int(global_q): local_q
        for local_q, global_q in enumerate(valid_rows.tolist())
    }
    local_source_splits = torch.tensor(
        [global_to_valid[int(q)] for q in source_splits.tolist()],
        dtype=torch.long,
    )

    temporal_trace = trace["temporal"]
    node_trace = temporal_trace["node"]

    for name, tensor in [
        ("temporal_normalized_query", temporal_trace["normalized"]),
        ("node_Q_projection", node_trace["q_projected"]),
        ("node_message_out", node_trace["message_out"]),
        ("node_gate", temporal_trace["node_gate"]),
        ("node_contribution", temporal_trace["node_contribution"]),
        ("tracklet_message_out", temporal_trace["tracklet"]["message_out"]),
        ("tracklet_gate", temporal_trace["tracklet_gate"]),
        ("tracklet_contribution", temporal_trace["tracklet_contribution"]),
        ("temporal_ffn_contribution", temporal_trace["ffn_contribution"]),
        ("temporal_fusion_final", temporal_trace["final"]),
    ]:
        selected = tensor[local_source_splits].detach().float().cpu().flatten(1)
        temporal_rows.append({
            "layer": layer,
            "quantity": name,
            **tensor_group_summary(selected),
            "abs_mean": float(selected.abs().mean()),
        })

    for term_name, tensor in [
        ("content_logits", node_trace["content_logits"]),
        ("relation_bias", node_trace["relation_bias"]),
        ("combined_logits", node_trace["combined_logits"]),
    ]:
        row = logit_term_summary(
            term_name,
            tensor,
            local_source_splits,
        )
        row["layer"] = layer
        temporal_rows.append(row)

    for weight_name, tensor in [
        ("content_only_weights", node_trace["content_weights"]),
        ("relation_only_weights", node_trace["relation_weights"]),
        ("production_combined_weights", node_trace["combined_weights"]),
    ]:
        selected = tensor[:, local_source_splits].detach().float().cpu()
        summary = distribution_group_summary(selected)
        temporal_rows.append({
            "layer": layer,
            "quantity": weight_name,
            **summary,
        })

temporal_df = pd.DataFrame(
    temporal_rows
)
temporal_df.to_csv(
    RUN_DIR / "source9_temporal_attention_decomposition.csv",
    index=False,
)
display(temporal_df)

## 11. Top historical nodes under content-only, relation-only, and production attention

If content-only assigns different nodes to different siblings but production does not, another attention term is washing out identity.

If content-only is already identical, the problem is earlier: query projections / memory keys are not producing identity-sensitive scores.

In [ ]:
top_rows = []
node_ids = state["temporal"].node_memory.node_ids
node_ids = (
    torch.arange(N, device=device)
    if node_ids is None
    else node_ids
)
node_times = state["temporal"].node_memory.time_offset
node_tracklets = state["temporal"].node_memory.tracklet_id

for trace in traces:
    layer = trace["layer"]
    valid_rows = trace["valid_rows"].tolist()
    global_to_valid = {
        int(global_q): local_q
        for local_q, global_q in enumerate(valid_rows)
    }
    local_splits = [
        global_to_valid[int(q)]
        for q in source_splits.tolist()
    ]

    node_trace = trace["temporal"]["node"]

    for mode, tensor in [
        ("content_only", node_trace["content_weights"]),
        ("relation_only", node_trace["relation_weights"]),
        ("combined", node_trace["combined_weights"]),
    ]:
        mean_weights = tensor.mean(dim=0)

        for slot, (global_q, local_q) in enumerate(
            zip(source_splits.tolist(), local_splits)
        ):
            weights = mean_weights[local_q]
            top = int(weights.argmax())
            top_rows.append({
                "layer": layer,
                "mode": mode,
                "query": int(global_q),
                "query_label": f"split{slot}",
                "top_node_index": top,
                "top_node_id": int(node_ids[top]),
                "time": int(round(float(node_times[top]))),
                "tracklet_id": int(node_tracklets[top]),
                "weight": float(weights[top]),
            })

top_df = pd.DataFrame(top_rows)
top_df.to_csv(
    RUN_DIR / "source9_temporal_top_nodes_by_term.csv",
    index=False,
)
display(top_df)

## 12. Residual contribution magnitudes

Even if attention differs, the temporal pathway can still have little effect if the gates make the residual message tiny relative to the query token.

This compares the actual residual magnitudes.

In [ ]:
contribution_rows = []

for trace in traces:
    layer = trace["layer"]
    valid_rows = trace["valid_rows"].tolist()
    global_to_valid = {
        int(global_q): local_q
        for local_q, global_q in enumerate(valid_rows)
    }
    local = torch.tensor(
        [global_to_valid[int(q)] for q in source_splits.tolist()],
        device=device,
        dtype=torch.long,
    )

    t = trace["temporal"]
    input_tokens = t["input"][local].float()

    quantities = {
        "input_token": input_tokens,
        "node_message_out": t["node"]["message_out"][local].float(),
        "node_gate": t["node_gate"][local].float(),
        "node_contribution": t["node_contribution"][local].float(),
        "tracklet_message_out": t["tracklet"]["message_out"][local].float(),
        "tracklet_gate": t["tracklet_gate"][local].float(),
        "tracklet_contribution": t["tracklet_contribution"][local].float(),
        "temporal_ffn_contribution": t["ffn_contribution"][local].float(),
    }

    base_norm = input_tokens.norm(dim=-1).mean().clamp_min(1e-12)

    for name, value in quantities.items():
        if "gate" in name:
            contribution_rows.append({
                "layer": layer,
                "quantity": name,
                "mean": float(value.mean()),
                "std": float(value.std(unbiased=False)),
                "mean_l2": float(value.norm(dim=-1).mean()),
                "relative_to_input_norm": float("nan"),
            })
        else:
            contribution_rows.append({
                "layer": layer,
                "quantity": name,
                "mean": float(value.mean()),
                "std": float(value.std(unbiased=False)),
                "mean_l2": float(value.norm(dim=-1).mean()),
                "relative_to_input_norm": float(
                    value.norm(dim=-1).mean()
                    / base_norm
                ),
            })

contribution_df = pd.DataFrame(
    contribution_rows
)
contribution_df.to_csv(
    RUN_DIR / "source9_temporal_residual_magnitudes.csv",
    index=False,
)
display(contribution_df)

## 13. Center divergence through the decoder

This keeps the previous center-separation diagnostic but aligns it with the exact sub-operation trace.

In [ ]:
center_rows = []

for trace in traces:
    layer = trace["layer"]
    before = (
        trace["reference_before"][0, source_splits]
        .float()
        * b["dref_um"][0].float()
    ).cpu()
    after = (
        trace["references_after"][0, source_splits]
        .float()
        * b["dref_um"][0].float()
    ).cpu()
    delta = (
        trace["center_delta"][0, source_splits]
        .float()
        * b["dref_um"][0].float()
    ).cpu()

    for label, tensor in [
        ("before", before),
        ("center_delta", delta),
        ("after", after),
    ]:
        pair = torch.pdist(tensor)
        center_rows.append({
            "layer": layer,
            "quantity": label,
            "pair_mean_um": float(pair.mean()) if pair.numel() else float("nan"),
            "pair_median_um": float(pair.median()) if pair.numel() else float("nan"),
            "pair_min_um": float(pair.min()) if pair.numel() else float("nan"),
            "pair_max_um": float(pair.max()) if pair.numel() else float("nan"),
        })

center_df = pd.DataFrame(center_rows)
center_df.to_csv(
    RUN_DIR / "source9_center_localization.csv",
    index=False,
)
display(center_df)

## 14. Counterfactual A — skip decoder self-attention before the layer-0 temporal read

This does **not** modify the model. It only asks:

> If layer 0 sent the QueryBuilder tokens directly into temporal memory, would the split slots become more temporally distinct?

A strong improvement here would directly implicate decoder self-attention as a homogenizer.

In [ ]:
@torch.no_grad()
def source_attention_summary_from_queries(
    query_tokens,
    qstate,
    layer,
    temporal,
    dref_um,
):
    valid = ~qstate.padding_mask
    flat = query_tokens[valid]
    refs_um = (
        qstate.references_cellscale
        * dref_um[:, None, None]
    )[valid]

    traced = trace_hierarchical_fusion(
        layer.temporal_fusion,
        flat,
        refs_um,
        temporal,
        dref_um,
    )

    valid_global = torch.nonzero(
        valid[0],
        as_tuple=False,
    ).flatten().tolist()
    global_to_local = {
        int(q): local
        for local, q in enumerate(valid_global)
    }
    local_split = torch.tensor(
        [global_to_local[int(q)] for q in source_splits.tolist()],
        dtype=torch.long,
    )

    result = {}
    for mode, tensor in [
        ("content", traced["node"]["content_weights"]),
        ("relation", traced["node"]["relation_weights"]),
        ("combined", traced["node"]["combined_weights"]),
    ]:
        selected = tensor[:, local_split].detach().float().cpu()
        result[mode] = distribution_group_summary(selected)

    result["q_projection"] = tensor_group_summary(
        traced["node"]["q_projected"][local_split]
        .detach()
        .float()
        .cpu()
    )
    return result

layer0 = model.query_decoder.layers[0]

production_after_self = traces[0]["after_self"]
skip_self_input = qstate.embeddings

prod_summary = source_attention_summary_from_queries(
    production_after_self,
    qstate,
    layer0,
    state["temporal"],
    b["dref_um"],
)
skip_summary = source_attention_summary_from_queries(
    skip_self_input,
    qstate,
    layer0,
    state["temporal"],
    b["dref_um"],
)

skip_rows = []
for variant, result in [
    ("production_after_self", prod_summary),
    ("skip_self_attention", skip_summary),
]:
    for term, values in result.items():
        skip_rows.append({
            "variant": variant,
            "term": term,
            **values,
        })

skip_df = pd.DataFrame(skip_rows)
skip_df.to_csv(
    RUN_DIR / "counterfactual_skip_self_attention.csv",
    index=False,
)
display(skip_df)

## 15. Counterfactual B — amplify only the already learned split-slot offsets

This is a **diagnostic perturbation**, not a proposed architecture.

For source-9 split queries only, we multiply the learned slot offset by:

`0, 1, 2, 4, 8, 16`

and then run the actual layer-0 self-attention followed by the actual temporal reader.

If temporal specialization appears only when the slot signal is magnified, the learned slot identity is probably too small relative to the shared component/query representation.

In [ ]:
@torch.no_grad()
def layer0_attention_with_slot_scale(scale):
    q_mod = qstate.embeddings.clone()

    for slot, q_index in enumerate(source_splits.tolist()):
        slot_vector = (
            model.query_builder.split_slot_embedding.weight[slot]
            .to(q_mod.dtype)
        )
        q_mod[0, q_index] = (
            q_mod[0, q_index]
            + (float(scale) - 1.0) * slot_vector
        )

    normed = layer0.self_norm(q_mod)
    self_message, _ = layer0.self_attn(
        normed,
        normed,
        normed,
        key_padding_mask=qstate.padding_mask,
        need_weights=False,
    )
    after_self = q_mod + self_message

    result = source_attention_summary_from_queries(
        after_self,
        qstate,
        layer0,
        state["temporal"],
        b["dref_um"],
    )

    split_rep = tensor_group_summary(
        q_mod[0, source_splits].float().cpu()
    )
    after_self_rep = tensor_group_summary(
        after_self[0, source_splits].float().cpu()
    )

    return {
        "scale": float(scale),
        "builder_cos_mean": split_rep["cos_mean"],
        "builder_pair_l2_mean": split_rep["pair_l2_mean"],
        "after_self_cos_mean": after_self_rep["cos_mean"],
        "after_self_pair_l2_mean": after_self_rep["pair_l2_mean"],
        "qproj_cos_mean": result["q_projection"]["cos_mean"],
        "content_weight_cos_mean": result["content"]["cos_mean"],
        "content_unique_top": result["content"]["unique_top_keys"],
        "combined_weight_cos_mean": result["combined"]["cos_mean"],
        "combined_js_proxy_pair_l2": result["combined"]["pair_l2_mean"],
        "combined_unique_top": result["combined"]["unique_top_keys"],
        "combined_entropy": result["combined"]["normalized_entropy_mean"],
    }

slot_scale_rows = [
    layer0_attention_with_slot_scale(scale)
    for scale in (0, 1, 2, 4, 8, 16)
]

slot_scale_df = pd.DataFrame(slot_scale_rows)
slot_scale_df.to_csv(
    RUN_DIR / "counterfactual_slot_scale.csv",
    index=False,
)
display(slot_scale_df)

ax = slot_scale_df.plot(
    x="scale",
    y=[
        "after_self_cos_mean",
        "qproj_cos_mean",
        "combined_weight_cos_mean",
    ],
    marker="o",
    figsize=(9, 4),
)
ax.set_title("Source-9 identity similarity vs diagnostic slot scale")
ax.set_ylabel("mean pairwise cosine")
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 16. Detection-GNN candidate-edge attention audit

The complete candidate graph gives each node roughly all other detections as incoming candidates.

The GAT is supposed to learn which relations matter. We now measure whether it actually does.

For each detection-GNN layer:

- normalized incoming-edge attention entropy;
- maximum incoming edge weight;
- attention mass on Trackastra-accepted edges;
- mass on same-frame edges;
- temporal-forward / temporal-reverse mass;
- the same metrics for nodes whose tracklet is associated with current source 9.

A normalized entropy near `1` indicates near-uniform attention across candidate neighbours.

In [ ]:
@torch.no_grad()
def initial_detection_embedding(model, b):
    scalar = model.graph_encoder.project_scalars(
        b["graph_x"]
    )
    if model.cfg.history.enabled:
        history = model.history_encoder(
            b["node_instance_grid"],
            b["node_history_valid"],
        )
        fused, history_gate = model.history_fusion(
            scalar,
            history,
            b["node_history_valid"],
        )
    else:
        fused = scalar
        history_gate = torch.zeros(
            (len(scalar), 1),
            device=scalar.device,
            dtype=scalar.dtype,
        )
    return fused, history_gate

@torch.no_grad()
def trace_gat_conv(conv, x, edge_index, edge_attr):
    n = x.shape[0]
    loops = torch.arange(
        n,
        device=x.device,
    )
    loop_index = torch.stack(
        [loops, loops],
        dim=0,
    )
    loop_attr = torch.zeros(
        (n, edge_attr.shape[-1]),
        device=x.device,
        dtype=edge_attr.dtype,
    )
    all_edge_index = torch.cat(
        [edge_index, loop_index],
        dim=1,
    )
    all_edge_attr = torch.cat(
        [edge_attr, loop_attr],
        dim=0,
    )

    src, dst = all_edge_index
    q = conv.q(x).view(
        n,
        conv.heads,
        conv.head_dim,
    )
    k = conv.k(x).view(
        n,
        conv.heads,
        conv.head_dim,
    )
    v = conv.v(x).view(
        n,
        conv.heads,
        conv.head_dim,
    )
    e = conv.edge(all_edge_attr).view(
        -1,
        conv.heads,
        conv.head_dim,
    )
    joint = F.leaky_relu(
        q[dst] + k[src] + e,
        negative_slope=0.2,
    )
    scores = (
        joint * conv.attn[None]
    ).sum(dim=-1) / math.sqrt(conv.head_dim)
    alpha = segment_softmax(
        scores,
        dst,
        n,
    ).float()

    msg = v[src] * alpha[..., None]
    out = torch.zeros(
        (n, conv.heads, conv.head_dim),
        device=x.device,
        dtype=x.dtype,
    )
    out.index_add_(
        0,
        dst,
        msg.to(out.dtype),
    )
    out = conv.out(
        out.reshape(n, conv.d_model)
    )

    return {
        "edge_index": all_edge_index,
        "edge_attr": all_edge_attr,
        "scores": scores.float(),
        "alpha": alpha,
        "message_out": out,
    }

def scatter_sum(values, index, n):
    out = torch.zeros(
        (n, values.shape[1]),
        device=values.device,
        dtype=values.dtype,
    )
    out.index_add_(0, index, values)
    return out

@torch.no_grad()
def gat_attention_metrics(trace, n, source9_node_mask):
    edge_index = trace["edge_index"]
    edge_attr = trace["edge_attr"]
    alpha = trace["alpha"]
    dst = edge_index[1]

    entropy_terms = -(
        alpha
        * alpha.clamp_min(1e-12).log()
    )
    entropy = scatter_sum(
        entropy_terms,
        dst,
        n,
    )
    degree = torch.bincount(
        dst,
        minlength=n,
    ).float().clamp_min(1)
    normalized_entropy = (
        entropy
        / degree.log().clamp_min(1e-8)[:, None]
    )

    max_weight = torch.full(
        (n, alpha.shape[1]),
        -torch.inf,
        device=alpha.device,
        dtype=alpha.dtype,
    )
    max_weight.scatter_reduce_(
        0,
        dst[:, None].expand_as(alpha),
        alpha,
        reduce="amax",
        include_self=True,
    )

    accepted = edge_attr[:, 14] > 0.5
    forward = edge_attr[:, 10] > 0.5
    reverse = edge_attr[:, 11] > 0.5
    same_frame = edge_attr[:, 13] > 0.5

    def mass(mask):
        weighted = alpha * mask[:, None].float()
        return scatter_sum(
            weighted,
            dst,
            n,
        )

    accepted_mass = mass(accepted)
    forward_mass = mass(forward)
    reverse_mass = mass(reverse)
    same_frame_mass = mass(same_frame)

    def summarize(mask):
        return {
            "nodes": int(mask.sum()),
            "normalized_entropy_mean": float(normalized_entropy[mask].mean()),
            "max_weight_mean": float(max_weight[mask].mean()),
            "accepted_mass_mean": float(accepted_mass[mask].mean()),
            "forward_mass_mean": float(forward_mass[mask].mean()),
            "reverse_mass_mean": float(reverse_mass[mask].mean()),
            "same_frame_mass_mean": float(same_frame_mass[mask].mean()),
        }

    all_mask = torch.ones(
        n,
        device=alpha.device,
        dtype=torch.bool,
    )
    result = {
        "all": summarize(all_mask),
        "source9": (
            summarize(source9_node_mask)
            if source9_node_mask.any()
            else None
        ),
    }
    return result, normalized_entropy, accepted_mass, max_weight

with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
    node_x, history_gate = initial_detection_embedding(
        model,
        b,
    )

tracklet_id = b["tracklet_id"]
best_component = b["best_current_component_id"]
source9_node_mask = (
    best_component[tracklet_id] == SOURCE_ID
)

gat_rows = []
gat_detail = []
x = node_x

for layer_index, block in enumerate(model.graph_encoder.layers):
    normed = block.norm1(x)
    trace = trace_gat_conv(
        block.attn,
        normed,
        b["graph_edge_index"],
        b["graph_edge_attr"],
    )
    metrics, normalized_entropy, accepted_mass, max_weight = gat_attention_metrics(
        trace,
        len(x),
        source9_node_mask,
    )

    for scope in ("all", "source9"):
        values = metrics[scope]
        if values is not None:
            gat_rows.append({
                "layer": layer_index,
                "scope": scope,
                **values,
            })

    x_after_attn = x + trace["message_out"]
    x = x_after_attn + block.ffn(
        block.norm2(x_after_attn)
    )

    gat_detail.append({
        "trace": trace,
        "normalized_entropy": normalized_entropy,
        "accepted_mass": accepted_mass,
        "max_weight": max_weight,
    })

gat_df = pd.DataFrame(gat_rows)
gat_df.to_csv(
    RUN_DIR / "detection_gnn_attention_summary.csv",
    index=False,
)
display(gat_df)

## 17. Top incoming GNN edges for source-9-related nodes

This makes the all-pairs GNN audit interpretable.

For each source-9-related destination node we show its strongest incoming relation by mean attention across heads.

In [ ]:
node_ids = b["node_ids"].detach().cpu()
node_times = b["node_time_offset"].detach().cpu()
tracklet_ids = b["tracklet_id"].detach().cpu()

gat_top_rows = []

for layer_index, detail in enumerate(gat_detail):
    trace = detail["trace"]
    edge_index = trace["edge_index"].detach().cpu()
    edge_attr = trace["edge_attr"].detach().cpu()
    alpha_mean = trace["alpha"].mean(dim=-1).detach().cpu()

    source9_destinations = torch.nonzero(
        source9_node_mask.detach().cpu(),
        as_tuple=False,
    ).flatten()

    for destination in source9_destinations.tolist():
        incoming = torch.nonzero(
            edge_index[1] == destination,
            as_tuple=False,
        ).flatten()
        if incoming.numel() == 0:
            continue
        best_edge = incoming[
            alpha_mean[incoming].argmax()
        ]
        source = int(edge_index[0, best_edge])
        attr = edge_attr[best_edge]

        gat_top_rows.append({
            "layer": layer_index,
            "dst_node_index": destination,
            "dst_node_id": int(node_ids[destination]),
            "dst_time": int(round(float(node_times[destination]))),
            "dst_tracklet": int(tracklet_ids[destination]),
            "src_node_index": source,
            "src_node_id": int(node_ids[source]),
            "src_time": int(round(float(node_times[source]))),
            "src_tracklet": int(tracklet_ids[source]),
            "attention": float(alpha_mean[best_edge]),
            "accepted": bool(attr[14] > 0.5),
            "dt": float(attr[0]),
            "distance_dref": float(attr[4]),
            "forward": bool(attr[10] > 0.5),
            "reverse": bool(attr[11] > 0.5),
            "same_frame": bool(attr[13] > 0.5),
        })

gat_top_df = pd.DataFrame(
    gat_top_rows
)
gat_top_df.to_csv(
    RUN_DIR / "source9_detection_gnn_top_edges.csv",
    index=False,
)
display(gat_top_df)

## 18. Automatic bottleneck summary

The rules below do not modify the model. They only convert the measured diagnostics into a concise localization report.

Read the numerical tables even if the automatic label is ambiguous.

In [ ]:
diagnosis = []

builder_row = representation_df[
    representation_df["stage"] == "query_builder_output"
].iloc[0]
after_self_row = representation_df[
    representation_df["stage"] == "L0_after_self"
].iloc[0]

layer0_temporal = temporal_df[
    temporal_df["layer"] == 0
]
content_row = layer0_temporal[
    layer0_temporal["quantity"] == "content_only_weights"
].iloc[0]
combined_row = layer0_temporal[
    layer0_temporal["quantity"] == "production_combined_weights"
].iloc[0]
qproj_row = layer0_temporal[
    layer0_temporal["quantity"] == "node_Q_projection"
].iloc[0]

if builder_row["cos_mean"] > 0.995:
    diagnosis.append(
        "QueryBuilder split siblings are already almost directionally identical; "
        "the learned slot cue is weak relative to the shared query representation."
    )

if after_self_row["cos_mean"] > builder_row["cos_mean"] + 0.002:
    diagnosis.append(
        "Layer-0 self-attention measurably increases sibling similarity, so it acts as a homogenizer."
    )
elif after_self_row["cos_mean"] >= 0.995:
    diagnosis.append(
        "Layer-0 output entering temporal memory is already highly homogeneous."
    )

if qproj_row["cos_mean"] >= 0.995:
    diagnosis.append(
        "The temporal node-reader Q projections remain nearly identical across split siblings."
    )

if content_row["cos_mean"] >= 0.995:
    diagnosis.append(
        "Even content-only Q·K attention is nearly identical; the collapse is not caused solely by the physical relation bias."
    )

if combined_row["cos_mean"] >= 0.995:
    diagnosis.append(
        "Production node-memory attention remains collapsed at layer 0."
    )

scale1 = slot_scale_df[
    slot_scale_df["scale"] == 1
].iloc[0]
scale8 = slot_scale_df[
    slot_scale_df["scale"] == 8
].iloc[0]
if scale8["combined_weight_cos_mean"] < scale1["combined_weight_cos_mean"] - 0.01:
    diagnosis.append(
        "Artificially amplifying the learned slot offsets makes temporal attention more distinct; "
        "slot-signal magnitude is a plausible bottleneck."
    )

if len(gat_df):
    gat_all = gat_df[
        (gat_df["layer"] == 0)
        & (gat_df["scope"] == "all")
    ]
    if len(gat_all) and float(gat_all.iloc[0]["normalized_entropy_mean"]) > 0.98:
        diagnosis.append(
            "Detection-GNN layer 0 also uses near-uniform candidate-edge attention; "
            "the all-pairs graph is information-preserving but not yet strongly selective."
        )

print("Notebook-16 localization report")
print("=" * 36)
for index, item in enumerate(diagnosis, start=1):
    print(f"{index}. {item}")

report = {
    "source_id": SOURCE_ID,
    "diagnosis": diagnosis,
    "builder": builder_row.to_dict(),
    "after_self": after_self_row.to_dict(),
    "layer0_content_attention": content_row.to_dict(),
    "layer0_combined_attention": combined_row.to_dict(),
    "layer0_q_projection": qproj_row.to_dict(),
}

with (
    RUN_DIR / "localization_summary.json"
).open("w", encoding="utf-8") as handle:
    json.dump(
        report,
        handle,
        indent=2,
        default=lambda x: (
            float(x)
            if isinstance(x, (np.floating, np.integer))
            else str(x)
        ),
    )

## 19. Compact comparison plot

This plot shows mean sibling cosine similarity through the sub-operations.

A sudden upward jump identifies a homogenizing block.

In [ ]:
plot_df = representation_df[
    representation_df["stage"].isin([
        "query_builder_output",
        "L0_after_self",
        "L0_after_temporal",
        "L0_after_spatial",
        "L0_final",
        "L1_after_self",
        "L1_after_temporal",
        "L1_after_spatial",
        "L1_final",
        "L2_after_self",
        "L2_after_temporal",
        "L2_after_spatial",
        "L2_final",
    ])
].copy()

ax = plot_df.plot(
    x="stage",
    y="cos_mean",
    marker="o",
    figsize=(12, 4),
)
ax.set_ylabel("mean pairwise cosine")
ax.set_title("Source-9 split identity through decoder sub-operations")
ax.grid(True, alpha=0.25)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 20. Save artifact index and free CUDA memory

The notebook does not alter repository code or model checkpoints.

In [ ]:
artifact_names = [
    "parameter_movement_step50_to_step85.csv",
    "source9_suboperation_representation.csv",
    "source9_self_attention.csv",
    "source9_temporal_attention_decomposition.csv",
    "source9_temporal_top_nodes_by_term.csv",
    "source9_temporal_residual_magnitudes.csv",
    "source9_center_localization.csv",
    "counterfactual_skip_self_attention.csv",
    "counterfactual_slot_scale.csv",
    "detection_gnn_attention_summary.csv",
    "source9_detection_gnn_top_edges.csv",
    "localization_summary.json",
]

print("Saved Notebook-16 diagnostics:")
for name in artifact_names:
    path = RUN_DIR / name
    print(" ", name, "OK" if path.exists() else "MISSING")

# Release the intentionally retained diagnostic tensors.
del traces, state, final_q
gc.collect()
torch.cuda.empty_cache()
print("CUDA cache cleared.")

# Interpretation guide

Use the outputs in this order.

### A. QueryBuilder output already has cosine ≈ 1

Then the split-slot identity is too small relative to the shared component/type representation before the decoder even starts.

Look at:

- `split-slot mean norm`;
- `split-slot mean pair distance`;
- `source-9 split tokens at QueryBuilder output`;
- step-50 → step-85 slot-embedding movement.

### B. QueryBuilder is distinct, but `L0_after_self` becomes much more similar

Then decoder self-attention is homogenizing sibling identities before temporal memory can use them.

The `skip_self_attention` counterfactual should also improve temporal-attention diversity.

### C. Q projections differ, but content-only softmax is still uniform

Then query differences are not aligned with useful key directions, or the Q·K logit scale is too weak.

Inspect:

- `node_Q_projection`;
- `content_logits`;
- `content_only_weights`.

### D. Content-only is distinct, relation-only is common, combined becomes identical

Then the common physical relation bias is dominating identity-sensitive content.

This is especially possible while siblings share exactly the same centroid.

### E. Temporal attention becomes distinct, but `after_temporal` / `after_spatial` / `final` becomes identical

Then the temporal reader works, but a downstream residual block homogenizes identity.

### F. Slot-scale counterfactual becomes sharply better

If increasing the already learned slot offset from `1×` to `4×/8×` produces clearly lower attention cosine and multiple top nodes, then the architecture has capacity but the slot identity is under-scaled or under-supervised.

### G. Detection-GNN normalized entropy ≈ 1

Then the all-pairs candidate graph is preserving evidence but the GAT itself is reading candidate relations diffusely.

Do not prune the graph merely from this result; first determine whether stronger learned selectivity / training pressure is needed.

---

The purpose of Notebook 16 is to identify the **first stage at which useful split identity disappears**. Only after that should the model architecture or objective be changed.